# Module 2 — Code Along: Lists, Dicts & Files (the bank-accounts story)

Module 1 gave us **one** account as a `dict`. Today we handle **many** accounts using the two data structures that matter — **list** and **dict** — and make them survive a restart by saving to **files** (CSV, JSON). The third structure, a **class**, is Module 3.

Section by section, one code cell per beat — run top to bottom (files are written to a temp dir, never the course `data/` dirs):

**Section 1** lists & CSV · **Section 2** dicts & JSON · **Section 3** organizing access via functions · **Section 4** logging

## Section 1 — Lists & CSV

The list is Python's ordered collection. We build it up — elements → slicing → length & looping → comprehension — then meet its real-world face: a **CSV** file, where every row reads back as a list.

In [ ]:
# 1.1 · List elements — a list holds many values in order; read by index (0-based; negatives from the end).
owners = ["Ada", "Lin", "Sam"]
print(owners[0])    # Ada  (first)
print(owners[-1])   # Sam  (last)
print(owners[1])    # Lin

In [ ]:
# 1.2 · Slicing — [start:stop] returns a sub-list; stop is excluded. Omit a side to run to the edge.
print(owners[0:2])  # ['Ada', 'Lin']
print(owners[1:])   # ['Lin', 'Sam']
print(owners[:2])   # ['Ada', 'Lin']

In [ ]:
# 1.3 · Length & looping — len() counts the items; a for loop walks them in order.
print(len(owners))  # 3
for name in owners:
    print(name)     # Ada / Lin / Sam

In [ ]:
# 1.4 · List comprehension — build a new list in one line; add an if to filter.
upper = [name.upper() for name in owners]
with_a = [n for n in owners if n.startswith("A")]
print(upper)        # ['ADA', 'LIN', 'SAM']  (transform)
print(with_a)       # ['Ada']                (filter)

In [ ]:
# 1.5 · A CSV row IS a list — write a small CSV, then read it with csv.reader (each row a list of strings).
import csv, tempfile, os
tmpdir = tempfile.mkdtemp()                 # demo files live here, not the course dirs
csv_path = os.path.join(tmpdir, "accounts.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "owner", "balance"])  # header row
    w.writerow([1, "Ada", 1500.0])
    w.writerow([2, "Lin", 800.0])

for row in csv.reader(open(csv_path)):
    print(row)    # ['id','owner','balance'] then ['1','Ada','1500.0'] ... every value is a str

In [ ]:
# 1.6 · Reading a CSV file — split the header off the data rows; or use DictReader to key by column.
rows = list(csv.reader(open(csv_path)))
header, data = rows[0], rows[1:]
print("header:", header)            # ['id', 'owner', 'balance']
print("first data row:", data[0])   # ['1', 'Ada', '1500.0']

# DictReader pairs each value with the header -> a dict per row (this bridges into Section 2).
for d in csv.DictReader(open(csv_path)):
    print(d)    # {'id': '1', 'owner': 'Ada', 'balance': '1500.0'}

In [ ]:
# 1.7 · Now with type hints — annotate collections & signatures so tools catch mistakes before you run.
owners_typed: list[str] = ["Ada", "Lin", "Sam"]
rows_typed: list[list[str]] = list(csv.reader(open(csv_path)))   # a CSV is a list of lists of str

def first_n(owners: list[str], n: int) -> list[str]:
    return owners[:n]

print(first_n(owners_typed, 2))   # ['Ada', 'Lin'] — hints document; they don't change behaviour
print(len(rows_typed), "rows")

In [ ]:
# 🔬 Explore — lists, slices, comprehensions, and CSV type-loss. Predict first.
owners = ["Ada", "Lin", "Sam"]
print(owners[-1], owners[1:])                      # 1. last item, and the tail slice?
print([n for n in owners if n.startswith("A")])    # 2. which names?

# 🔧 TRY — uncomment, predict, run:
# owners[5]                    # index past the end — what error?
# owners[1:99]                 # slice past the end — error, or clamped?
# import csv, io
# row = next(csv.reader(io.StringIO("1,Ada,1500.0")))
# row[2] + 100                 # a CSV value + a number — what breaks, and why?

### ✅ Check yourself — Section 1
- [ ] `owners[5]` raised but `owners[1:99]` didn't. Why do index and slice differ?
  _hint: a slice clamps to the edges; an index demands the position exists._
- [ ] After `csv.reader`, why did `row[2] + 100` fail?
  _hint: every CSV cell reads back as a ___._
- [ ] Write the comprehension for "names longer than 3 letters."
  _hint: `[n for n in owners if len(n) > 3]`._

## Section 2 — Dicts & JSON

The dict stores named fields looked up by key. We build it up — access → looping → common functions → nesting — then persist it: a dict maps exactly to **JSON**, so it round-trips to disk and back as a Python dict.

In [ ]:
# 2.1 · Dict access — look up by key (not position). [key] raises if missing; .get(key) is safe.
acct = {"id": 1, "owner": "Ada", "balance": 1500.0}
print(acct["owner"])      # Ada
print(acct.get("phone"))  # None — no KeyError

In [ ]:
# 2.2 · Dict looping — keys, values, or both with .items().
for key in acct:
    print("key:", key)
for value in acct.values():
    print("value:", value)
for key, value in acct.items():
    print(key, "=", value)

In [ ]:
# 2.3 · Common dict ops — assign/update to add or overwrite; `in` to test; pop to remove.
acct["balance"] = 1600.0           # update one field
acct.update({"tier": "gold"})      # add / merge fields
print("owner" in acct)             # True
print(acct.pop("tier"))            # gold  (removed)
print(acct)                        # back to id / owner / balance

In [ ]:
# 2.4 · Nested dicts — values can themselves be lists or dicts; reach in by chaining keys.
acct = {"owner": "Ada", "address": {"city": "Pune"}, "tags": ["primary", "online"]}
print(acct["address"]["city"])   # Pune
print(acct["tags"][0])           # primary

In [ ]:
# 2.5 · JSON round-trip — a dict maps exactly to JSON; dump to disk, load straight back as a dict.
import json
json_path = os.path.join(tmpdir, "acct.json")
json.dump(acct, open(json_path, "w"), indent=2)   # dict -> file
back = json.load(open(json_path))                 # file -> dict
print(back)
print(back["address"]["city"])   # Pune — same shape, immediately a Python dict
print(type(back))                # <class 'dict'>

In [ ]:
# 2.6 · Now with type hints — type a dict by key/value; mixed values stay vague (the hint you want a class).
balances: dict[str, float] = {"Ada": 1500.0, "Lin": 800.0}                 # uniform -> precise
account: dict[str, object] = {"id": 1, "owner": "Ada", "balance": 1500.0}  # mixed -> dict[str, object]
print(balances["Ada"])     # 1500.0
print(account["owner"])    # Ada
# the vague dict[str, object] is the signal a fixed-field record wants a class (Module 3)

In [ ]:
# 🔬 Explore — dict ops, nesting, and the JSON-keys gotcha. Predict first.
import json, io
acct = {"owner": "Ada", "address": {"city": "Pune"}, "tags": ["primary"]}
print(acct["address"]["city"])                     # 1. reach into the nested dict?

store = {1: {"owner": "Ada"}}                       # an int key
back = json.load(io.StringIO(json.dumps(store)))   # round-trip through JSON
print(list(back.keys()))                           # 2. still an int key?

# 🔧 TRY — predict, run:
# back[1]                # look up with the int 1 — KeyError? why?
# back["1"]              # and with the string?
# acct.get("missing"), acct.pop("missing")   # which of these is safe on a missing key?

### ✅ Check yourself — Section 2
- [ ] After a JSON round-trip, the int key `1` came back as ___. Why?
  _hint: JSON object keys can only be one type._
- [ ] `.get("missing")` vs `["missing"]` vs `.pop("missing")` — which raise?
  _hint: only `.get` is safe by default._
- [ ] Why is `dict[str, object]` for a mixed account a "smell"?
  _hint: it signals a fixed-field record wants a ___ (Module 3)._

## Section 3 — Organizing dict access via functions

Keep all accounts in one dict keyed by id, and reach it only through `insert` / `update` / `fetch` functions. Those functions hide *where* the data lives — a JSON file today, a database tomorrow.

In [ ]:
# 3.1 · A dict as the store — accounts keyed by id give instant (O(1)) lookup by account number.
accounts = {}                                   # {id: account}
accounts[1] = {"id": 1, "owner": "Ada", "balance": 1500.0}
accounts[2] = {"id": 2, "owner": "Lin", "balance": 800.0}
print(accounts[1])    # lookup by id, no scanning

In [ ]:
# 3.2 · insert & update via functions — one place owns writes, so every caller behaves the same.
def insert(store, acct):
    store[acct["id"]] = acct

def update(store, id, **changes):
    store[id].update(changes)

insert(accounts, {"id": 3, "owner": "Sam", "balance": 40.0})
update(accounts, 3, balance=90.0)
print(accounts[3])    # {'id': 3, 'owner': 'Sam', 'balance': 90.0}

In [ ]:
# 3.3 · fetch via a function — one place to validate / raise on a missing id.
def fetch(store, id):
    if id not in store:
        raise LookupError(f"no account id={id}")
    return store[id]

print(fetch(accounts, 1)["owner"])   # Ada
try:
    fetch(accounts, 99)
except LookupError as err:
    print("handled:", err)           # handled: no account id=99

In [ ]:
# 3.4 · The seam — the functions hide WHERE data lives. Today: a JSON file. Tomorrow: a DB / warehouse.
import json
store_path = os.path.join(tmpdir, "accounts.json")
json.dump(accounts, open(store_path, "w"), indent=2)   # persist the whole store
accounts = json.load(open(store_path))                 # reload from disk

# GOTCHA: JSON object keys are always strings -> ids come back as "1","2","3", not 1,2,3.
print(list(accounts.keys()))             # ['1', '2', '3']
print(fetch(accounts, "1")["owner"])     # Ada — same fetch(), data now comes from disk
# tomorrow: accounts = db.query("SELECT ..."); insert()/fetch() on top are unchanged

In [ ]:
# 3.5 · Now with type hints — typed signatures make the storage layer self-documenting.
def insert(store: dict[int, dict], acct: dict) -> None:
    store[acct["id"]] = acct

def fetch(store: dict[int, dict], id: int) -> dict:
    if id not in store:
        raise LookupError(f"no account id={id}")
    return store[id]

typed_store: dict[int, dict] = {}
insert(typed_store, {"id": 1, "owner": "Ada", "balance": 1500.0})
print(fetch(typed_store, 1)["owner"])   # Ada

In [ ]:
# 🔬 Explore — a keyed store reached only through functions. Predict first.
def insert(store, acct):
    store[acct["id"]] = acct

def fetch(store, id):
    if id not in store:
        raise LookupError(f"no id={id}")
    return store[id]

s = {}
insert(s, {"id": 1, "owner": "Ada"})
print(fetch(s, 1)["owner"])        # 1. who?

# 🔧 TRY — predict, run:
# fetch(s, 99)                                          # missing id — what does the seam do?
# insert(s, {"id": 1, "owner": "Zoe"}); fetch(s, 1)     # same id again — added, or overwritten?

### ✅ Check yourself — Section 3
- [ ] Why keep the store as `{id: acct}` instead of a list of accounts?
  _hint: lookup cost — O(1) vs scanning every item._
- [ ] `insert` with an existing id overwrote it. Where would you add a duplicate-guard?
  _hint: one place owns writes — inside `insert`._
- [ ] "JSON today, a database tomorrow" — what stays unchanged when storage swaps?
  _hint: every caller of `insert`/`fetch`._

## Section 4 — Logging

`print` is for throwaway scripts; real code uses `logging` — levels let you dial detail up or down. We add it to the storage functions so every access leaves an audit trail.

In [ ]:
# 4.1 · Logging vs print — levels (DEBUG < INFO < WARNING < ERROR) dial detail without deleting lines.
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
log = logging.getLogger("bank")
log.debug("hidden at INFO level")   # not shown
log.info("storage ready")           # shown
log.warning("low balance!")         # shown

In [ ]:
# 4.2 · Logging the storage — one line per op; now every fetch leaves an audit trail.
def fetch(store, id):
    log.info("fetch id=%s", id)     # %s args, not an f-string (lazy, structured)
    if id not in store:
        log.warning("missing id=%s", id)
        raise LookupError(f"no account id={id}")
    return store[id]

store = json.load(open(store_path))
print(fetch(store, "1")["owner"])   # logs: fetch id=1 ; returns Ada
try:
    fetch(store, "99")              # logs: fetch id=99 ; missing id=99
except LookupError:
    pass

In [ ]:
# 🔬 Explore — logging levels. Predict what prints.
import logging
logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s", force=True)
log = logging.getLogger("bank")
log.info("loaded 3 accounts")     # 1. shown at WARNING level?
log.warning("low balance")        # 2. and this?

# 🔧 TRY — predict, run:
# logging.basicConfig(level=logging.DEBUG, force=True); log.debug("cache hit")   # now visible?
# log.info("fetch id=%s", 7)     # %s style vs an f-string — why prefer %s here?

### ✅ Check yourself — Section 4
- [ ] At `level=WARNING`, why did `log.info(...)` stay silent?
  _hint: only messages at or above the threshold are emitted._
- [ ] Order these low→high: INFO, ERROR, DEBUG, WARNING.
  _hint: DEBUG is the chattiest, ERROR the loudest._
- [ ] Why `log.info("id=%s", id)` rather than an f-string?
  _hint: the arg is only formatted if the line is actually emitted._

## Next: Module 3

We now hold **many** accounts as `dict`s in a `list`/keyed store, persist them to **CSV/JSON**, reach them through `insert`/`update`/`fetch` functions, and **log** every access.

**Next we wrap that data *and* its functions into one object — a `class`** (`BankAccount`). The dict's fields become attributes; the storage functions become methods. That's the most powerful "data structure" of all — and Module 3.